# Step 5 — Feature Engineering
### Credit Card Underwriting Pipeline

---

## What is Feature Engineering?

> **Feature Engineering = creating new, more informative inputs from existing columns.**
>
> Raw data rarely tells the whole story. A model given only `annual_income` and `debt_amount`
> separately might miss that their *ratio* (Debt-to-Income) is far more predictive of credit risk.
> Feature engineering closes that gap.

---

## Topics Covered in This Notebook

| Section | Topic | Why It Matters |
|---------|-------|----------------|
| 5.1 | WoE & IV | Measure predictive power; credit industry standard encoding |
| 5.2 | Ratio / Interaction Features | Capture relationships between columns |
| 5.3 | Log Transforms | Reduce skew in income/asset columns |
| 5.4 | Categorical Encoding | Encode remaining categoricals |
| 5.5 | Multicollinearity Check | Remove redundant, highly correlated features |
| 5.6 | IV-based Feature Selection | Keep only features with meaningful predictive power |
| 5.7 | Final Feature List | The clean set of features for modelling |

---

## The WoE/IV Framework — Credit Scorecard Industry Standard

WoE (Weight of Evidence) and IV (Information Value) come from logistic regression-based
credit scorecard methodology, used in banking for decades.

**WoE formula:**
$$\text{WoE}_i = \ln\left(\frac{\%\text{Events}_i}{\%\text{Non-Events}_i}\right)$$

Where:
- **Event** = Approved (target = 1)
- **Non-Event** = Declined (target = 0)
- Each bin `i` is a range of the feature values

**IV formula:**
$$\text{IV} = \sum_i (\%E_i - \%NE_i) \times \text{WoE}_i$$

**IV Interpretation Table:**

| IV Range | Predictive Power |
|----------|-----------------|
| < 0.02 | Useless — drop this feature |
| 0.02 – 0.10 | Weak — marginal utility |
| 0.10 – 0.30 | Medium — good feature |
| > 0.30 | Strong — likely a top feature |
| > 0.50 | Suspicious — check for target leakage |


In [ ]:
# ── Re-run Setup + Steps 1-4 ──────────────────────────────────────────────────
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')

DATA_PATH   = 'cc_underwriting_5k_stratified11.csv'
IGNORE_COLS = ['applicant_id', 'target_approved', 'target_credit_limit_assigned']
SEED = 42

df_raw = pd.read_csv(DATA_PATH)
numeric_cols = [c for c in df_raw.select_dtypes(include='number').columns if c not in IGNORE_COLS]
cat_cols     = [c for c in df_raw.select_dtypes(include=['object','string']).columns if c not in IGNORE_COLS]

def missing_report(df):
    m=df.isnull().sum(); p=m/len(df)*100
    r=pd.DataFrame({'missing_count':m,'missing_pct':p.round(2)})
    return r[r['missing_count']>0].sort_values('missing_pct',ascending=False)

miss = missing_report(df_raw)
drop_cols = miss[miss['missing_pct']>50].index.tolist()
df = df_raw.drop(columns=drop_cols)
numeric_cols = [c for c in numeric_cols if c in df.columns]
cat_cols     = [c for c in cat_cols     if c in df.columns]

for col in numeric_cols:
    if df[col].isnull().sum()>0: df[col].fillna(df[col].median(), inplace=True)
for col in cat_cols:
    if df[col].isnull().sum()>0: df[col].fillna(df[col].mode()[0], inplace=True)

df['target'] = (df['target_approved'] == 'Yes').astype(int)
print(f'Data ready: {df.shape}')

## 5.1 — WoE & IV Calculation

### Step-by-Step: How WoE/IV is Computed

1. **Bin the feature** (10 quantile bins for numeric; raw values for categorical)
2. **Count events (Approved) and non-events (Declined) in each bin**
3. **Compute % events and % non-events** relative to totals
4. **Compute WoE** = ln(%events / %non-events) per bin
5. **Compute IV** = sum over all bins of (%events - %non-events) × WoE

> **Why WoE encoding?**
> After computing WoE, you can replace the original feature values with their WoE values.
> This has 3 benefits:
> - All features are on the same scale (log-odds)
> - Handles both numeric and categorical features uniformly
> - Aligns features with the target variable's scale (logistic regression-friendly)


In [ ]:
def compute_woe_iv(df, feature, target='target', bins=10, cat=False):
    """
    Compute WoE table and scalar IV for a single feature.

    Parameters:
        df      : DataFrame containing the feature and target
        feature : column name to analyse
        target  : binary target column name (1=event, 0=non-event)
        bins    : number of quantile bins for numeric features
        cat     : True if feature is categorical (use raw values as bins)

    Returns:
        (woe_table, iv_scalar)
        woe_table: DataFrame with WoE and IV per bin
        iv_scalar: float — total Information Value
    """
    # Total events and non-events across the whole dataset
    total_events     = df[target].sum()              # total Approved
    total_non_events = len(df) - total_events        # total Declined

    if cat:
        # Categorical: group by the raw category values
        groups = df.groupby(feature)[target]
    else:
        # Numeric: bin into quantiles (equal-frequency buckets)
        try:
            df = df.copy()
            df['_bin'] = pd.qcut(df[feature], q=bins, duplicates='drop')
        except Exception:
            df['_bin'] = pd.cut(df[feature], bins=bins)  # fallback to equal-width
        groups = df.groupby('_bin')[target]

    # Aggregate: sum = events (approved), count = total in bin
    tbl = groups.agg(['sum', 'count'])

    # Normalise column names robustly
    col_map = {}
    for c in tbl.columns:
        if 'sum' in c.lower():   col_map[c] = 'events'
        elif 'count' in c.lower(): col_map[c] = 'total'
    tbl = tbl.rename(columns=col_map)

    tbl['non_events'] = tbl['total'] - tbl['events']  # Declined count per bin

    # % of all events that fall in this bin
    tbl['pct_events']     = tbl['events']     / total_events
    # % of all non-events that fall in this bin
    tbl['pct_non_events'] = tbl['non_events'] / total_non_events

    # Smooth zeros — log(0) is undefined; add tiny epsilon
    eps = 1e-6
    tbl['woe'] = np.log((tbl['pct_events'] + eps) / (tbl['pct_non_events'] + eps))
    tbl['iv']  = (tbl['pct_events'] - tbl['pct_non_events']) * tbl['woe']

    iv_total = tbl['iv'].sum()   # sum IV across all bins
    result   = tbl.reset_index()

    # Rename the index column to the feature name
    if result.columns[0] != feature:
        result = result.rename(columns={result.columns[0]: feature})

    return result, iv_total


def iv_label(iv):
    """Return human-readable strength label for an IV value."""
    if iv < 0.02:  return 'Useless'
    if iv < 0.10:  return 'Weak'
    if iv < 0.30:  return 'Medium'
    return 'Strong'


print('WoE/IV functions defined')

# --- Example: compute WoE for fico_score ---
woe_tbl, iv_val = compute_woe_iv(df.copy(), 'fico_score', cat=False, bins=10)
print(f'IV for fico_score: {iv_val:.4f}  ({iv_label(iv_val)})')
print()
print(woe_tbl[['fico_score','events','non_events','pct_events','pct_non_events','woe','iv']].to_string())

In [ ]:
# Visualise WoE pattern for fico_score
fig, ax = plt.subplots(figsize=(13, 4))

x_labels = [str(b) for b in woe_tbl['fico_score']]
bar_colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in woe_tbl['woe']]

ax.bar(x_labels, woe_tbl['woe'], color=bar_colors, edgecolor='white', width=0.7)
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('FICO Score Bin', fontsize=11)
ax.set_ylabel('WoE', fontsize=11)
ax.set_title('WoE by FICO Score Bin\n(Green = more approvals, Red = more declines)',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  Positive WoE = this bin has proportionally MORE approvals than declines')
print('  Negative WoE = this bin has proportionally MORE declines than approvals')
print('  As FICO increases, WoE increases — higher FICO = more likely approved (expected!)')

In [ ]:
# Compute IV for ALL features
MAX_CAT_CARDINALITY = 30
selected_cats = [c for c in cat_cols if df[c].nunique() <= MAX_CAT_CARDINALITY]
selected_nums = numeric_cols[:60]

iv_results = []

for col in selected_nums:
    try:
        _, iv = compute_woe_iv(df.copy(), col, cat=False)
        iv_results.append({'feature': col, 'iv': iv, 'type': 'numeric'})
    except Exception:
        pass

for col in selected_cats:
    try:
        _, iv = compute_woe_iv(df.copy(), col, cat=True)
        iv_results.append({'feature': col, 'iv': iv, 'type': 'categorical'})
    except Exception:
        pass

iv_df = pd.DataFrame(iv_results).sort_values('iv', ascending=False).reset_index(drop=True)
iv_df['strength'] = iv_df['iv'].apply(iv_label)

print(f'IV computed for {len(iv_df)} features')
print()
print('Top 20 features by IV:')
print(iv_df.head(20).to_string(index=False))

In [ ]:
# IV bar chart — visual ranking of all features by predictive power
TOP_N = 30
top_iv = iv_df.head(TOP_N)

color_map = {'Useless':'#bdc3c7','Weak':'#f39c12','Medium':'#2ecc71','Strong':'#2980b9'}
colors    = top_iv['strength'].map(color_map)

fig, ax = plt.subplots(figsize=(14, 9))
ax.barh(top_iv['feature'][::-1], top_iv['iv'][::-1],
        color=colors[::-1], edgecolor='white')

# Threshold lines
for x, lbl, clr in [(0.02,'Weak (0.02)','#e74c3c'),
                     (0.10,'Medium (0.10)','#e67e22'),
                     (0.30,'Strong (0.30)','#27ae60')]:
    ax.axvline(x=x, color=clr, linestyle='--', linewidth=1.3, label=lbl)

ax.set_xlabel('Information Value (IV)', fontsize=12)
ax.set_title(f'Top {TOP_N} Features by Information Value\n'
             '(Blue=Strong, Green=Medium, Orange=Weak, Grey=Useless)',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5.2 — Ratio & Interaction Features

> **Why create interaction features?**
>
> The model can only see one column at a time when making a split.
> An interaction feature like `income / requested_limit` encodes the relationship
> between two columns as a single new column, making the signal immediately visible.
>
> **Financial ratios are also interpretable** — a bank underwriter already uses
> Debt-to-Income, Loan-to-Value, etc. We are giving the model the same tools.


In [ ]:
df_fe = df.copy()
eps = 1e-6   # small constant to prevent division by zero

# ── Financial Stress Ratios ────────────────────────────────────────────────────
# How big is the requested limit relative to income? A $50k limit on a $30k income is risky.
df_fe['income_to_limit_ratio']    = df_fe['annual_income'] / (df_fe['requested_credit_limit'] + eps)

# What fraction of monthly income goes to expenses? High ratio = financial stress.
df_fe['expenses_to_income_ratio'] = df_fe['total_monthly_expenses'] / (df_fe['monthly_income'] + eps)

# Are savings a meaningful buffer relative to income?
df_fe['savings_to_income_ratio']  = df_fe['savings_account_balance'] / (df_fe['annual_income'] + eps)

# Higher assets-to-liabilities = healthier balance sheet
df_fe['assets_to_liabilities']    = df_fe['total_assets'] / (df_fe['total_liabilities'] + eps)

# ── Credit Health Features ─────────────────────────────────────────────────────
# Average across all 4 bureau scores = more stable credit score estimate
df_fe['bureau_score_mean'] = df_fe[['fico_score','equifax_score',
                                     'experian_score','transunion_score']].mean(axis=1)

# Standard deviation across bureaus = how consistent are the bureaus?
# High std = bureaus disagree = possible error or thin file
df_fe['bureau_score_std']  = df_fe[['fico_score','equifax_score',
                                     'experian_score','transunion_score']].std(axis=1)

df_fe['credit_history_yrs']       = df_fe['credit_history_length_months'] / 12

# High utilization AND many inquiries together = risky behaviour
df_fe['utilization_x_inquiries']  = df_fe['credit_utilization_ratio'] * df_fe['hard_inquiries_last_12mo']

# More derogatory marks + higher DTI = compounded risk
df_fe['derogatory_x_dti']         = df_fe['derogatory_marks_count'] * df_fe['debt_to_income_ratio']

# ── Income & Employment ────────────────────────────────────────────────────────
df_fe['monthly_net_income']       = df_fe['monthly_income'] - df_fe['total_monthly_expenses']
df_fe['income_per_dependent']     = df_fe['annual_income'] / (df_fe['dependents_count'] + 1)
df_fe['income_yrs_employed']      = df_fe['annual_income'] * df_fe['years_employed']

# ── Risk Flag Aggregation ──────────────────────────────────────────────────────
# Encode each Yes/No risk flag as 1/0, then sum them
risk_flags = ['prior_default_flag','prior_bankruptcy_flag','high_risk_industry_flag',
              'recent_address_change_flag','recent_employment_change_flag',
              'multiple_applications_flag','thin_file_flag','no_hit_flag']

for f in risk_flags:
    if f in df_fe.columns:
        df_fe[f + '_enc'] = (df_fe[f] == 'Yes').astype(int)

risk_enc_cols = [f + '_enc' for f in risk_flags if f + '_enc' in df_fe.columns]
df_fe['total_risk_flags'] = df_fe[risk_enc_cols].sum(axis=1)

# ── Age Interactions ───────────────────────────────────────────────────────────
df_fe['age_squared'] = df_fe['age'] ** 2         # capture non-linear age effect
df_fe['age_x_fico']  = df_fe['age'] * df_fe['fico_score']  # older + better FICO = lower risk

NEW_FEATURES = [
    'income_to_limit_ratio','expenses_to_income_ratio','savings_to_income_ratio',
    'assets_to_liabilities','bureau_score_mean','bureau_score_std','credit_history_yrs',
    'utilization_x_inquiries','derogatory_x_dti','monthly_net_income',
    'income_per_dependent','income_yrs_employed','total_risk_flags',
    'age_squared','age_x_fico'
]

print(f'Created {len(NEW_FEATURES)} new engineered features')
print()
print(df_fe[NEW_FEATURES].describe().T[['mean','std','min','max']].round(3).to_string())

## 5.3 — Log Transforms for Skewed Features

> **Why log-transform?**
>
> Income distributions are heavily right-skewed: most people earn $30k–$100k,
> but a few earn $5M+. This extreme right tail distorts the model:
> - It may try to fit the millionaire and get the middle class wrong
> - It can confuse distance-based methods and regularised linear models
>
> **log1p(x) = log(x + 1)** is safe for zero values (log(0) is undefined but log(1) = 0).
> After transform, the distribution looks much more bell-shaped.
>
> **Note:** Random Forest is tree-based so it is scale-invariant — it does not strictly require
> log transforms. However, log transforms still help by reducing the impact of extreme outliers
> on tree splits.


In [ ]:
# Columns known to be heavily right-skewed from EDA
SKEWED_COLS = ['annual_income','monthly_income','net_worth','total_assets',
               'savings_account_balance','requested_credit_limit','total_liabilities']

fig, axes = plt.subplots(2, len(SKEWED_COLS), figsize=(22, 7))

for i, col in enumerate(SKEWED_COLS):
    if col not in df_fe.columns:
        continue

    # Apply log1p transform: log(x + 1)
    # np.clip ensures no negative values enter the log (clip to 0 minimum)
    log_col = col + '_log'
    df_fe[log_col] = np.log1p(np.clip(df_fe[col], 0, None))

    # Top row: original distribution
    axes[0, i].hist(df_fe[col].dropna(), bins=40, color='#e74c3c', alpha=0.8, edgecolor='white')
    axes[0, i].set_title(f'{col}\n(original)', fontsize=8, fontweight='bold')

    # Bottom row: log-transformed
    axes[1, i].hist(df_fe[log_col].dropna(), bins=40, color='#2ecc71', alpha=0.8, edgecolor='white')
    axes[1, i].set_title(f'{col}\n(log1p)', fontsize=8, fontweight='bold')

plt.suptitle('Log Transform: Before (Red) vs After (Green)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

LOG_FEATURES = [c + '_log' for c in SKEWED_COLS if c + '_log' in df_fe.columns]
print(f'Created {len(LOG_FEATURES)} log-transformed features: {LOG_FEATURES}')

## 5.4 — Encode Remaining Categoricals

> At this stage we encode the remaining categorical columns that were NOT yet encoded
> in Step 4 (for example, any columns added by engineered features).


In [ ]:
# Encode categoricals with <= 30 unique values
cat_for_model = [c for c in cat_cols if df_fe[c].nunique() <= 30]

le = LabelEncoder()
for col in cat_for_model:
    # Overwrite or create the _enc column
    col_enc = col + '_enc'
    df_fe[col_enc] = le.fit_transform(df_fe[col].astype(str))

CAT_ENC_FEATURES = [c + '_enc' for c in cat_for_model]
print(f'Label-encoded {len(CAT_ENC_FEATURES)} categorical features')

## 5.5 — Multicollinearity Check

> **Multicollinearity = two or more features that are very highly correlated with each other.**
>
> **Why is this a problem?**
> - In linear models (logistic regression), multicollinearity makes coefficients unstable and
>   uninterpretable (the model cannot tell which feature is "responsible")
> - In Random Forest, it is less critical because trees are not affected the same way,
>   but it still wastes computation and can dilute feature importance scores
>
> **Our rule:** If two features have |correlation| > 0.95, drop one of them.
> We keep the one with higher IV (more predictive).
>
> **Visual aid:** Heatmap where dark red = very positively correlated, dark blue = negatively correlated.


In [ ]:
# Build full feature list before correlation filter
ALL_FEATURES = list(dict.fromkeys(    # dict.fromkeys preserves order and removes duplicates
    numeric_cols + NEW_FEATURES + LOG_FEATURES + CAT_ENC_FEATURES
))
ALL_FEATURES = [f for f in ALL_FEATURES if f in df_fe.columns]
print(f'Total candidate features before correlation filter: {len(ALL_FEATURES)}')

# Compute correlation matrix
X_raw = df_fe[ALL_FEATURES].copy()
corr_matrix = X_raw.corr().abs()   # absolute values — we care about magnitude, not direction

# Extract the upper triangle only (avoid double-counting pairs)
# np.triu with k=1 sets the diagonal and lower triangle to False
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find columns where ANY pair correlation exceeds the threshold
CORR_THRESH = 0.95
to_drop_corr = [col for col in upper_tri.columns if any(upper_tri[col] > CORR_THRESH)]
print(f'Dropping {len(to_drop_corr)} highly correlated features:')
for c in to_drop_corr[:10]:
    print(f'  {c}')
if len(to_drop_corr) > 10:
    print(f'  ... and {len(to_drop_corr)-10} more')

In [ ]:
# Visualise correlation heatmap for top 20 features (post-filter)
MIN_IV    = 0.02
iv_selected = iv_df[iv_df['iv'] >= MIN_IV]['feature'].tolist()
engineered  = NEW_FEATURES + LOG_FEATURES

FEATURES_AFTER_IV = [f for f in ALL_FEATURES if f in iv_selected or f in engineered]
FINAL_FEATURES    = [f for f in FEATURES_AFTER_IV if f not in to_drop_corr]

print(f'Features after IV filter        : {len(FEATURES_AFTER_IV)}')
print(f'Features after correlation filter: {len(FINAL_FEATURES)}')

# Heatmap: only top 20 to keep it readable
top20 = FINAL_FEATURES[:20]
corr20 = df_fe[top20].corr()
mask   = np.triu(np.ones_like(corr20, dtype=bool))   # mask upper triangle (duplicate)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr20, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Correlation Matrix — Top 20 Final Features\n'
             '(Green=positive, Red=negative; closer to 1.0 = more correlated)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print(f'FINAL feature count ready for modelling: {len(FINAL_FEATURES)}')

## Step 5 Complete — Feature Engineering Summary

**What we built:**

| Feature Type | Count | Examples |
|-------------|-------|---------|
| Raw numeric | ~60 | fico_score, annual_income, age |
| Ratio/interaction | 15 | income_to_limit_ratio, derogatory_x_dti |
| Log-transformed | 7 | annual_income_log, net_worth_log |
| Label-encoded categoricals | varies | employment_status_enc |

**Feature Selection Pipeline:**
1. Computed IV for all features → `iv_df`
2. Kept features with IV >= 0.02 (or engineered)
3. Dropped one from each pair with correlation > 0.95
4. Result: `FINAL_FEATURES` — the clean input set for training

**Next:** `06_Train_Test_Split.ipynb` — Stratified split, SMOTE, scaling
